# 03 - Avaliacao base x LoRA no Google Colab

Notebook executavel em Colab limpo com GPU. Ele baixa os pesos publicados no Hub e salva evidencias novas no Google Drive, sem substituir a entrega local.

## 1. Controles e caminhos

Ajuste `SELECTED_LORA_CONFIG` para avaliar `config_a` ou `config_b`. Por padrao, a melhor configuracao ainda nao e assumida; use a configuracao que voce quer comparar com o modelo base.

In [ ]:
# Colab bootstrap: run this cell first in a clean GPU runtime.
import os
import subprocess
from pathlib import Path

IN_COLAB = True
PROJECT_REPO_URL = "https://github.com/EngIaCeub/atelie-generativo-felipe-santiago.git"
COLAB_ROOT = Path("/content/atelie_generativo")
PROJECT_DIR = COLAB_ROOT / "repo"
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["git", "clone", PROJECT_REPO_URL, str(PROJECT_DIR)] if not PROJECT_DIR.exists() else ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_DIR)
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    token = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
except ImportError:
    raise RuntimeError("Este notebook deve ser executado no Google Colab.")
print("Projeto Colab:", PROJECT_DIR)


In [ ]:
# Colab bootstrap: run this cell first in a clean GPU runtime.
import os
import subprocess
from pathlib import Path

IN_COLAB = True
PROJECT_REPO_URL = "https://github.com/EngIaCeub/atelie-generativo-felipe-santiago.git"
COLAB_ROOT = Path("/content/atelie_generativo")
PROJECT_DIR = COLAB_ROOT / "repo"
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["git", "clone", PROJECT_REPO_URL, str(PROJECT_DIR)] if not PROJECT_DIR.exists() else ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_DIR)
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    token = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
except ImportError:
    raise RuntimeError("Este notebook deve ser executado no Google Colab.")
print("Projeto Colab:", PROJECT_DIR)


In [12]:
from __future__ import annotations

import csv
import datetime as dt
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont

NOTEBOOK_REVISION = "evaluation-colab-2026-07-11-v1"
RUN_GENERATION = True
RUN_CLIPSCORE = True
RUN_MEMORIZATION = True
RUN_HUMAN_ANALYSIS = True
SELECTED_LORA_CONFIG = "config_b"
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5
IMAGE_SIZE = 512
NEGATIVE_PROMPT = "low quality, blurry, watermark, signature, text, logo"

ROOT = Path.cwd()
if not (ROOT / "config/project.json").exists() and (ROOT.parent / "config/project.json").exists():
    ROOT = ROOT.parent
CONFIG_PATH = ROOT / "config/project.json"
RESULTS_DIR = Path("/content/drive/MyDrive/atelie_generativo_felipe_santiago/avaliacao_colab")
GENERATED_DIR = RESULTS_DIR / "geradas"
MEMORY_DIR = RESULTS_DIR / "memorizacao_imagens"
PROMPTS_PATH = ROOT / "dados/prompts_avaliacao.txt"
METADATA_PATH = ROOT / "dados/metadata.jsonl"
TRAINING_CSV = ROOT / "resultados/treino/experimentos.csv"
from huggingface_hub import hf_hub_download
_hub_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
LORA_WEIGHTS = Path(hf_hub_download(_hub_config["hugging_face"]["lora_repo_id"], "pytorch_lora_weights.safetensors", token=os.environ.get("HF_TOKEN")))
LORA_DIR = LORA_WEIGHTS.parent

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
trigger = config["style"]["trigger_token"]
base_model_id = config["models"]["base_diffusion"]
clip_model_id = config["models"]["clip"]
generation_seed = int(config["evaluation"].get("generation_seed", 2026))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook revision:", NOTEBOOK_REVISION)
print("Projeto:", config["project_name"])
print("Modelo base:", base_model_id)
print("LoRA selecionada:", SELECTED_LORA_CONFIG)
print("Pesos LoRA:", LORA_WEIGHTS)
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert LORA_WEIGHTS.exists(), f"Pesos LoRA ausentes: {LORA_WEIGHTS}"


Notebook revision: evaluation-local-2026-07-11-v1
Projeto: atelie-generativo-felipe-santiago
Modelo base: stable-diffusion-v1-5/stable-diffusion-v1-5
LoRA selecionada: config_b
Pesos LoRA: c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\treino\local\config_b\pytorch_lora_weights.safetensors
CUDA disponivel: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 2. Funcoes auxiliares

As funcoes abaixo mantem execucao reprodutivel, criam grades e evitam depender do diretorio atual do kernel.

In [2]:
def run(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    printable = " ".join(str(part) for part in cmd)
    print("$", printable)
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, text=True, check=check)

def read_csv(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

def write_csv(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def slug(text: str, limit: int = 64) -> str:
    keep = []
    for char in text.lower():
        if char.isalnum():
            keep.append(char)
        elif char in {" ", "-", "_"}:
            keep.append("-")
    out = "".join(keep).strip("-")
    while "--" in out:
        out = out.replace("--", "-")
    return out[:limit] or "item"

def image_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def torch_generator(seed: int):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return torch.Generator(device=device).manual_seed(seed)

def latest_completed_config_rows() -> pd.DataFrame:
    rows = read_csv(TRAINING_CSV)
    return pd.DataFrame([row for row in rows if row.get("status") in {"concluido", "completed"}])

def make_grid(rows: list[dict[str, object]], output_path: Path) -> None:
    images = []
    for row in rows:
        base_img = Image.open(row["base_path"]).convert("RGB")
        lora_img = Image.open(row["lora_path"]).convert("RGB")
        images.append((row, base_img, lora_img))
    cell_w, cell_h = IMAGE_SIZE, IMAGE_SIZE
    label_h = 96
    grid = Image.new("RGB", (cell_w * 2, (cell_h + label_h) * len(images)), "white")
    draw = ImageDraw.Draw(grid)
    try:
        font = ImageFont.truetype("arial.ttf", 18)
        small = ImageFont.truetype("arial.ttf", 14)
    except OSError:
        font = ImageFont.load_default()
        small = ImageFont.load_default()
    for idx, (row, base_img, lora_img) in enumerate(images):
        y = idx * (cell_h + label_h)
        draw.rectangle([0, y, cell_w * 2, y + label_h], fill=(245, 245, 245))
        draw.text((10, y + 8), f"Prompt {row['prompt_id']}: {row['prompt'][:110]}", fill=(0, 0, 0), font=small)
        draw.text((10, y + 58), "BASE", fill=(0, 0, 0), font=font)
        draw.text((cell_w + 10, y + 58), f"LoRA {SELECTED_LORA_CONFIG}", fill=(0, 0, 0), font=font)
        grid.paste(base_img.resize((cell_w, cell_h)), (0, y + label_h))
        grid.paste(lora_img.resize((cell_w, cell_h)), (cell_w, y + label_h))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    grid.save(output_path)


## 3. Prompts fixos

Esta celula congela 6 prompts em `dados/prompts_avaliacao.txt` quando o arquivo ainda esta vazio ou contem apenas comentario.

In [3]:
default_prompts = [
    f"{trigger}, um lobo-guara caminhando entre capim dourado no Cerrado, xilogravura digital de alto contraste",
    f"{trigger}, uma seriema sobre uma pedra ao amanhecer, hachuras pretas e composicao grafica artesanal",
    f"{trigger}, flores de pequi e folhas retorcidas do Cerrado em padrao ornamental de xilogravura",
    f"{trigger}, uma cachoeira pequena cercada por buritis e pedras, paleta reduzida em preto e marfim",
    f"{trigger}, tamandua-bandeira atravessando uma trilha de terra vermelha, contornos espessos e textura manual",
    f"{trigger}, vaso de barro com frutos nativos do Cerrado sobre mesa simples, gravura digital com sombras hachuradas",
]
existing = [
    line.strip()
    for line in PROMPTS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
] if PROMPTS_PATH.exists() else []
if len(existing) != 6:
    PROMPTS_PATH.write_text("\n".join(default_prompts) + "\n", encoding="utf-8")
    prompts = default_prompts
    print("Prompts de avaliacao criados:", PROMPTS_PATH)
else:
    prompts = existing
    print("Prompts de avaliacao existentes mantidos:", PROMPTS_PATH)
for i, prompt in enumerate(prompts, start=1):
    print(f"{i}. {prompt}")
assert len(prompts) == 6, "A avaliacao exige exatamente 6 prompts fixos."


Prompts de avaliacao existentes mantidos: c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\dados\prompts_avaliacao.txt
1. flpxilobr, um lobo-guara caminhando entre capim dourado no Cerrado, xilogravura digital de alto contraste
2. flpxilobr, uma seriema sobre uma pedra ao amanhecer, hachuras pretas e composicao grafica artesanal
3. flpxilobr, flores de pequi e folhas retorcidas do Cerrado em padrao ornamental de xilogravura
4. flpxilobr, uma cachoeira pequena cercada por buritis e pedras, paleta reduzida em preto e marfim
5. flpxilobr, tamandua-bandeira atravessando uma trilha de terra vermelha, contornos espessos e textura manual
6. flpxilobr, vaso de barro com frutos nativos do Cerrado sobre mesa simples, gravura digital com sombras hachuradas


## 4. Carregamento dos pipelines

Carrega um pipeline base e um pipeline LoRA usando os pesos locais gerados no notebook 02. O token Hugging Face deve estar no ambiente quando o modelo base exigir autenticacao, mas o valor nunca e exibido.

In [4]:
from diffusers import DPMSolverMultistepScheduler, StableDiffusionPipeline

if RUN_GENERATION:
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Carregando pipeline base...")
    base_pipe = StableDiffusionPipeline.from_pretrained(
        base_model_id,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    base_pipe.scheduler = DPMSolverMultistepScheduler.from_config(base_pipe.scheduler.config)
    base_pipe = base_pipe.to(device)
    base_pipe.set_progress_bar_config(disable=False)

    print("Carregando pipeline LoRA...")
    lora_pipe = StableDiffusionPipeline.from_pretrained(
        base_model_id,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    lora_pipe.scheduler = DPMSolverMultistepScheduler.from_config(lora_pipe.scheduler.config)
    lora_pipe.load_lora_weights(str(LORA_DIR), weight_name="pytorch_lora_weights.safetensors")
    lora_pipe = lora_pipe.to(device)
    lora_pipe.set_progress_bar_config(disable=False)

    print("Pipelines carregados em", device)
else:
    print("RUN_GENERATION=False; pipelines nao carregados.")


c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Carregando pipeline base...


Loading pipeline components...: 100%|██████████| 6/6 [00:04<00:00,  1.38it/s]


Carregando pipeline LoRA...


Loading pipeline components...: 100%|██████████| 6/6 [00:04<00:00,  1.36it/s]
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Pipelines carregados em cuda


## 5. Geracao base x LoRA e grade 6 x 2

Gera imagens individuais para os 6 prompts com a mesma seed para base e LoRA, depois salva uma grade comparativa.

In [5]:
comparison_rows = []
if RUN_GENERATION:
    for idx, prompt in enumerate(prompts, start=1):
        seed = generation_seed + idx
        filename = f"prompt_{idx:02d}_{slug(prompt, 36)}"
        base_path = GENERATED_DIR / f"{filename}_base.png"
        lora_path = GENERATED_DIR / f"{filename}_{SELECTED_LORA_CONFIG}.png"
        if not base_path.exists():
            image = base_pipe(
                prompt=prompt,
                negative_prompt=NEGATIVE_PROMPT,
                width=IMAGE_SIZE,
                height=IMAGE_SIZE,
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=torch_generator(seed),
            ).images[0]
            image.save(base_path)
        if not lora_path.exists():
            image = lora_pipe(
                prompt=prompt,
                negative_prompt=NEGATIVE_PROMPT,
                width=IMAGE_SIZE,
                height=IMAGE_SIZE,
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=torch_generator(seed),
            ).images[0]
            image.save(lora_path)
        comparison_rows.append({
            "prompt_id": idx,
            "prompt": prompt,
            "seed": seed,
            "base_path": str(base_path),
            "lora_path": str(lora_path),
            "lora_config": SELECTED_LORA_CONFIG,
        })
    write_csv(
        RESULTS_DIR / "geracoes.csv",
        comparison_rows,
        ["prompt_id", "prompt", "seed", "base_path", "lora_path", "lora_config"],
    )
    make_grid(comparison_rows, RESULTS_DIR / "grade_comparativa.png")
    print("Grade salva em:", RESULTS_DIR / "grade_comparativa.png")
else:
    comparison_rows = read_csv(RESULTS_DIR / "geracoes.csv")
    print("RUN_GENERATION=False; usando geracoes existentes:", len(comparison_rows))
assert len(comparison_rows) == 6, "Sao necessarias 6 linhas de geracao para a grade comparativa."


100%|██████████| 30/30 [01:19<00:00,  2.64s/it]


Grade salva em: c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\resultados\avaliacao\grade_comparativa.png


## 6. CLIPScore

Calcula similaridade texto-imagem com CLIP para cada imagem gerada e exporta resultados individuais e medias por modelo.

In [6]:
if RUN_CLIPSCORE:
    from transformers import CLIPModel, CLIPProcessor

    clip_device = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model = CLIPModel.from_pretrained(clip_model_id).to(clip_device)
    clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
    clip_model.eval()

    clip_rows = []
    with torch.no_grad():
        for row in comparison_rows:
            prompt = row["prompt"]
            for model_name, image_path in [("base", row["base_path"]), (SELECTED_LORA_CONFIG, row["lora_path"] )]:
                image = Image.open(image_path).convert("RGB")
                inputs = clip_processor(text=[prompt], images=[image], return_tensors="pt", padding=True).to(clip_device)
                outputs = clip_model(**inputs)
                image_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
                text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
                score = float((image_embeds * text_embeds).sum().item() * 100.0)
                clip_rows.append({
                    "prompt_id": row["prompt_id"],
                    "prompt": prompt,
                    "modelo": model_name,
                    "imagem": image_path,
                    "clipscore": f"{score:.6f}",
                })
    write_csv(RESULTS_DIR / "clipscore.csv", clip_rows, ["prompt_id", "prompt", "modelo", "imagem", "clipscore"])
    summary = (
        pd.DataFrame(clip_rows)
        .assign(clipscore=lambda df: df["clipscore"].astype(float))
        .groupby("modelo", as_index=False)["clipscore"]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )
    summary.to_csv(RESULTS_DIR / "clipscore_resumo.csv", index=False, encoding="utf-8")
    print(summary)
else:
    clip_rows = read_csv(RESULTS_DIR / "clipscore.csv")
    print("RUN_CLIPSCORE=False; usando CLIPScore existente:", len(clip_rows))
assert len(clip_rows) >= 12, "CLIPScore precisa de pelo menos 12 linhas: 6 prompts x 2 modelos."


c:\Users\felip\OneDrive\Documentos\Engenharia de Inteligência Artificial\Modelos Multimodais\sistematizacao\atelie-generativo-felipe-santiago\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\felip\.cache\huggingface\hub\models--openai--clip-vit-base-patch16. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(m

   index    modelo       mean       std        min        max
0      0      base  26.786041  3.690527  22.258356  32.782775
1      1  config_b  23.273084  3.042174  19.779481  28.824174


## 7. Teste de memorizacao

Gera 10 casos proximos das captions revisadas e compara cada imagem gerada com o dataset por similaridade CLIP imagem-imagem. Isso e um indicador de risco, nao prova definitiva de copia.

In [9]:
metadata_records = [
    json.loads(line)
    for line in METADATA_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
memorization_prompts = [record["text"] for record in metadata_records[:10]]
assert len(memorization_prompts) == 10, "Sao necessarios 10 prompts de memorizacao."

if RUN_MEMORIZATION:
    # Libera VRAM para a geracao das imagens de memorizacao.
    if "base_pipe" in globals():
        del base_pipe
    if "clip_model" in globals():
        clip_model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    clip_device = "cpu"

    if "clip_model" not in globals():
        from transformers import CLIPModel, CLIPProcessor

        clip_model = CLIPModel.from_pretrained(clip_model_id).to(clip_device)
        clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
        clip_model.eval()

    def get_clip_image_embeddings(inputs):
        features = clip_model.get_image_features(**inputs)
        # transformers 5 retorna BaseModelOutputWithPooling.
        return features if isinstance(features, torch.Tensor) else features.pooler_output

    train_image_paths = [ROOT / record["file_name"] for record in metadata_records]
    train_images = [Image.open(path).convert("RGB") for path in train_image_paths]

    with torch.no_grad():
        train_inputs = clip_processor(
            images=train_images,
            return_tensors="pt",
            padding=True,
        ).to(clip_device)
        train_embeds = get_clip_image_embeddings(train_inputs)
        train_embeds = train_embeds / train_embeds.norm(dim=-1, keepdim=True)

    memory_rows = []
    for idx, prompt in enumerate(memorization_prompts, start=1):
        seed = generation_seed + 1000 + idx
        generated_path = MEMORY_DIR / f"mem_{idx:02d}_{SELECTED_LORA_CONFIG}.png"

        if RUN_GENERATION and not generated_path.exists():
            image = lora_pipe(
                prompt=prompt,
                negative_prompt=NEGATIVE_PROMPT,
                width=IMAGE_SIZE,
                height=IMAGE_SIZE,
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=torch_generator(seed),
            ).images[0]
            image.save(generated_path)

        generated_image = Image.open(generated_path).convert("RGB")

        with torch.no_grad():
            gen_inputs = clip_processor(
                images=[generated_image],
                return_tensors="pt",
                padding=True,
            ).to(clip_device)
            gen_embed = get_clip_image_embeddings(gen_inputs)
            gen_embed = gen_embed / gen_embed.norm(dim=-1, keepdim=True)
            similarities = (gen_embed @ train_embeds.T).squeeze(0).detach().cpu().numpy()

        best_idx = int(np.argmax(similarities))
        max_similarity = float(similarities[best_idx])

        if max_similarity >= 0.95:
            risk = "alto"
        elif max_similarity >= 0.90:
            risk = "medio"
        else:
            risk = "baixo"

        memory_rows.append(
            {
                "caso": idx,
                "prompt": prompt,
                "imagem_gerada": str(generated_path),
                "imagem_treino_mais_proxima": str(train_image_paths[best_idx]),
                "similaridade_clip_imagem": f"{max_similarity:.6f}",
                "risco_copia": risk,
                "observacao": (
                    "Revisar visualmente pares com risco medio/alto; "
                    "similaridade CLIP nao prova copia."
                ),
            }
        )

    write_csv(
        RESULTS_DIR / "memorizacao.csv",
        memory_rows,
        [
            "caso",
            "prompt",
            "imagem_gerada",
            "imagem_treino_mais_proxima",
            "similaridade_clip_imagem",
            "risco_copia",
            "observacao",
        ],
    )
else:
    memory_rows = read_csv(RESULTS_DIR / "memorizacao.csv")
    print("RUN_MEMORIZATION=False; usando memorizacao existente:", len(memory_rows))

assert len(memory_rows) >= 10, "A avaliacao exige 10 casos de memorizacao."
pd.DataFrame(memory_rows).head(10)

100%|██████████| 30/30 [00:06<00:00,  4.85it/s]


,caso,prompt,imagem_gerada,imagem_treino_mais_proxima,similaridade_clip_imagem,risco_copia,observacao
0,1,"flpxilobr, frutos de araticum do brejo em vege...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.734186,baixo,Revisar visualmente pares com risco medio/alto...
1,2,"flpxilobr, arvore tortuosa de casca grossa tip...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.814113,baixo,Revisar visualmente pares com risco medio/alto...
2,3,"flpxilobr, queda d'agua entre rochas e mata de...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.911940,medio,Revisar visualmente pares com risco medio/alto...
3,4,"flpxilobr, cachos de buriti e folhas de palmei...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.869037,baixo,Revisar visualmente pares com risco medio/alto...
4,5,"flpxilobr, bromelia nativa com folhas radiais ...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.883139,baixo,Revisar visualmente pares com risco medio/alto...
5,6,"flpxilobr, beija-flor pousado em ramo fino, co...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.901010,medio,Revisar visualmente pares com risco medio/alto...
6,7,"flpxilobr, bacurau camuflado sobre tronco e fo...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.894620,baixo,Revisar visualmente pares com risco medio/alto...
7,8,"flpxilobr, galhos secos e copa de arvore do Ce...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.919870,medio,Revisar visualmente pares com risco medio/alto...
8,9,"flpxilobr, cacho de flores de Byrsonima entre ...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.889988,baixo,Revisar visualmente pares com risco medio/alto...
9,10,"flpxilobr, flor clara isolada sobre fundo escu...",c:\Users\felip\OneDrive\Documentos\Engenharia ...,c:\Users\felip\OneDrive\Documentos\Engenharia ...,0.891547,baixo,Revisar visualmente pares com risco medio/alto...


## 8. Avaliacao humana cega

Esta celula prepara a chave e o formulario cego. Ela so gera `avaliacao_humana.csv` completo quando houver respostas reais em `resultados/avaliacao/avaliacao_humana_respostas.csv`.

Formato esperado de respostas reais: `id_avaliador,sample_id,preferencia,aderencia_estilo,qualidade,comentario`. Nao preencha avaliadores ficticios.

In [13]:
random.seed(generation_seed)
blind_rows = []
key_rows = []
for row in comparison_rows:
    options = [("base", row["base_path"]), (SELECTED_LORA_CONFIG, row["lora_path"])]
    random.shuffle(options)
    sample_id = f"p{int(row['prompt_id']):02d}"
    blind_rows.append({
        "sample_id": sample_id,
        "prompt": row["prompt"],
        "imagem_a": options[0][1],
        "imagem_b": options[1][1],
        "pergunta_preferencia": "Qual imagem representa melhor o prompt e o estilo proposto? Responda A, B ou empate.",
        "aderencia_estilo_1a5": "",
        "qualidade_1a5": "",
        "comentario": "",
    })
    key_rows.extend([
        {"sample_id": sample_id, "opcao": "A", "modelo": options[0][0], "imagem": options[0][1]},
        {"sample_id": sample_id, "opcao": "B", "modelo": options[1][0], "imagem": options[1][1]},
    ])
write_csv(
    RESULTS_DIR / "formulario_avaliacao_humana.csv",
    blind_rows,
    ["sample_id", "prompt", "imagem_a", "imagem_b", "pergunta_preferencia", "aderencia_estilo_1a5", "qualidade_1a5", "comentario"],
)
write_csv(RESULTS_DIR / "avaliacao_humana_chave.csv", key_rows, ["sample_id", "opcao", "modelo", "imagem"])

responses_path = RESULTS_DIR / "avaliacao_humana_respostas.csv"
output_path = RESULTS_DIR / "avaliacao_humana.csv"
if RUN_HUMAN_ANALYSIS and responses_path.exists():
    responses = read_csv(responses_path)
    raters = {row.get("id_avaliador", "").strip() for row in responses if row.get("id_avaliador", "").strip()}
    assert len(raters) >= int(config["evaluation"].get("minimum_external_raters", 5)), "Avaliacao humana requer 5+ avaliadores externos reais."
    write_csv(
        output_path,
        responses,
        ["id_avaliador", "sample_id", "preferencia", "aderencia_estilo", "qualidade", "comentario"],
    )
    print("Respostas humanas anonimizadas importadas:", len(responses))
else:
    if not output_path.exists() or output_path.stat().st_size == 0:
        write_csv(output_path, [], ["id_avaliador", "sample_id", "preferencia", "aderencia_estilo", "qualidade", "comentario"])
    print("Formulario cego preparado:", RESULTS_DIR / "formulario_avaliacao_humana.csv")
    print("Chave separada preparada:", RESULTS_DIR / "avaliacao_humana_chave.csv")
    print("Portao humano pendente: coletar 5+ avaliadores externos reais antes de marcar esta etapa como concluida.")


Respostas humanas anonimizadas importadas: 30


## 9. Sintese tecnica e validacao

Consolida artefatos automaticos e roda as validacoes locais. A validacao de etapa ainda deve falhar enquanto a avaliacao humana real nao for importada.

In [14]:
summary = {
    "data": dt.datetime.now(dt.timezone.utc).isoformat(),
    "selected_lora_config": SELECTED_LORA_CONFIG,
    "base_model": base_model_id,
    "clip_model": clip_model_id,
    "generation_seed": generation_seed,
    "prompts_file": str(PROMPTS_PATH),
    "comparison_grid": str(RESULTS_DIR / "grade_comparativa.png"),
    "clipscore_csv": str(RESULTS_DIR / "clipscore.csv"),
    "memorization_csv": str(RESULTS_DIR / "memorizacao.csv"),
    "human_evaluation_status": "pendente_respostas_reais",
    "notes": [
        "Nao interpretar CLIPScore isoladamente como qualidade final.",
        "Revisar visualmente casos de memorizacao com maior similaridade.",
        "Nao declarar avaliacao humana concluida sem 5+ pessoas externas reais.",
    ],
}
(RESULTS_DIR / "resumo_avaliacao.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))

os.environ.setdefault("PYTHONIOENCODING", "utf-8")
run([sys.executable, str(ROOT / "scripts/check_secrets.py")], cwd=ROOT)
run([sys.executable, str(ROOT / "scripts/project_status.py")], cwd=ROOT)
# A linha abaixo deve aprovar somente depois das respostas humanas reais.
run([sys.executable, str(ROOT / "scripts/validate_project.py"), "--stage", "evaluation"], cwd=ROOT, check=False)


{
  "data": "2026-07-11T14:46:08.040208+00:00",
  "selected_lora_config": "config_b",
  "base_model": "stable-diffusion-v1-5/stable-diffusion-v1-5",
  "clip_model": "openai/clip-vit-base-patch16",
  "generation_seed": 2026,
  "prompts_file": "c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie-generativo-felipe-santiago\\dados\\prompts_avaliacao.txt",
  "comparison_grid": "c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie-generativo-felipe-santiago\\resultados\\avaliacao\\grade_comparativa.png",
  "clipscore_csv": "c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie-generativo-felipe-santiago\\resultados\\avaliacao\\clipscore.csv",
  "memorization_csv": "c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie

CompletedProcess(args=['c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie-generativo-felipe-santiago\\.venv\\Scripts\\python.exe', 'c:\\Users\\felip\\OneDrive\\Documentos\\Engenharia de Inteligência Artificial\\Modelos Multimodais\\sistematizacao\\atelie-generativo-felipe-santiago\\scripts\\validate_project.py', '--stage', 'evaluation'], returncode=0)

## Portao humano

A etapa de avaliacao so pode ser considerada concluida depois de importar respostas anonimizadas reais de pelo menos 5 avaliadores externos em `resultados/avaliacao/avaliacao_humana_respostas.csv` e reexecutar a celula de avaliacao humana.